# UserLogin Data Cleaning and loading
---
## Objective :
Our main objective is to generate a report in which we have to display the maximum number of unique employee login its date and also display the average login count with the following conditions:
1. we only want login data
2. we have to filter out the days on which the login count < 5 (this will be taken care within data_analysis.ipynb notebook)
3. the data should be displayed for every financial year
4. all the mentioned details must be visualized

## Data Files and their variables
|File Name|Variable Name|Dataframe|
|-|-|-|
|UserLogs.txt|ul01|df1|
|UserLogs 02092025.txt|ul02|df2

In [ ]:
# importing the necessary libraries
import pandas as pd
import re

In [ ]:
# loading the file in a variable name

with open(r'Z:\\UserLogin_data_analysis\UserLogs_Data\UserLogs.txt','r') as file:
    ul01=file.read()
with open(r'Z:\\UserLogin_data_analysis\UserLogs_Data\UserLogs 02092025.txt','r') as file:
    ul02=file.read()

In [ ]:
# Split the loaded content into lines
lines = ul01.splitlines()

# Parse each line into structured fields
records = []
for line in lines:
    record = {}

    # Match each field safely
    timestamp_match = re.search(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} [APM]{2}', line)
    session_match = re.search(r'Session:([^\s]+)', line)
    ip_match = re.search(r'Client IP:([^\s]+)', line)
    user_match = re.search(r'Username:([^\s]+)', line)
    event_match = re.search(r'Event:([^\s]+)', line)
    type_match = re.search(r'LoginSessionType:([^\s]+)', line)

    # Assign values only if match is found
    record['Timestamp'] = timestamp_match.group() if timestamp_match else None
    record['Session ID'] = session_match.group(1) if session_match else None
    record['Client IP'] = ip_match.group(1) if ip_match else None
    record['Username'] = user_match.group(1) if user_match else None
    record['Event'] = event_match.group(1) if event_match else None
    record['LoginSessionType'] = type_match.group(1) if type_match else None

    records.append(record)

# Convert to DataFrame
df1 = pd.DataFrame(records)

# Preview the result
df1.head()


In [ ]:
import pandas as pd
import re

# Split the content into lines
lines = ul02.splitlines()

# Prepare list for structured records
records = []

for line in lines:
    line = line.strip()
    if not line:
        continue  # Skip empty lines

    # Match simple log format: Event Date Time Username
    match = re.match(r'(LogIn|LogOut)\s+(\d{4}-\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2})\s+(\S+)', line)
    if match:
        record = {
            'Event': match.group(1),
            'Timestamp': f"{match.group(2)} {match.group(3)}",
            'Username': match.group(4)
        }
        records.append(record)

# Convert to DataFrame
df2 = pd.DataFrame(records)

# Preview the result
df2.head()


In [ ]:
df1.columns

In [ ]:
df1.drop(columns=['Session ID', 'Client IP', 'LoginSessionType'],inplace=True)

In [ ]:
df1['Timestamp']=pd.to_datetime(df1['Timestamp'])

In [ ]:
df2['Timestamp']=pd.to_datetime(df2['Timestamp'])

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
df1_login=df1[df1['Event'].str.lower()=='login'].drop(columns='Event')
df1_login['Date']=pd.to_datetime(df1_login['Timestamp']).dt.date
df1_login['Time']=pd.to_datetime(df1_login['Timestamp']).dt.time
df1_login.head()

In [ ]:
df2_login=df2[df2['Event'].str.lower()=='login'].drop(columns='Event')
df2_login['Date']=pd.to_datetime(df2_login['Timestamp']).dt.date
df2_login['Time']=pd.to_datetime(df2_login['Timestamp']).dt.time
df2_login.head()

In [ ]:
df1_login['Timestamp'].dt.date.max(),\
df1_login['Timestamp'].dt.date.min()

In [ ]:
df2_login['Timestamp'].dt.date.max(),\
df2_login['Timestamp'].dt.date.min()

In [ ]:
latest_date = df2_login['Timestamp'].max()
df_temp = df1_login[df1_login['Timestamp'] > latest_date]

In [ ]:
df_temp.head()

In [ ]:
df_final=pd.concat([df2_login,df_temp],ignore_index=True)
df_final.head()

In [ ]:
df_final['Timestamp'].dt.date.max(),\
df_final['Timestamp'].dt.date.min()

In [ ]:
df_final.to_csv('path',index=False) # substitute the path value